In [1]:
import duckdb

# 1. Quanti milioni di righe ci sono nel file?
print("Conteggio totale dei viaggi nel Parquet:")
query_conteggio = "SELECT COUNT(*) AS totale_viaggi FROM 'yellow_tripdata_2025-01.parquet'"
display(duckdb.query(query_conteggio).df())

# 2. La JOIN in tempo reale tra Parquet (Fatti) e CSV (Dimensioni)
print("\nTop 10 Quartieri per numero di partenze:")
query_join = """
    SELECT 
        zone.Borough AS Distretto,
        zone.Zone AS Quartiere,
        COUNT(*) AS Numero_Partenze
    FROM 'yellow_tripdata_2025-01.parquet' AS viaggi
    JOIN 'taxi_zone_lookup.csv' AS zone
      ON viaggi.PULocationID = zone.LocationID
    GROUP BY zone.Borough, zone.Zone
    ORDER BY Numero_Partenze DESC
    LIMIT 10;
"""
display(duckdb.query(query_join).df())

Conteggio totale dei viaggi nel Parquet:


,totale_viaggi
0,3475226



Top 10 Quartieri per numero di partenze:


,Distretto,Quartiere,Numero_Partenze
0,Manhattan,Midtown Center,169977
1,Manhattan,Upper East Side South,163703
2,Manhattan,Upper East Side North,155647
3,Queens,JFK Airport,146137
4,Manhattan,Times Sq/Theatre District,125829
5,Manhattan,Penn Station/Madison Sq West,119131
6,Manhattan,Midtown East,117930
7,Manhattan,Lincoln Square East,110585
8,Manhattan,Upper West Side South,96614
9,Manhattan,Midtown North,95906


In [2]:
%pip install duckdb pandas pyarrow

   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/13.1 MB 1.9 MB/s eta 0:00:07
    --------------------------------------- 0.3/13.1 MB 4.1 MB/s eta 0:00:04
   - -------------------------------------- 0.4/13.1 MB 3.8 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/13.1 MB 4.6 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/13.1 MB 5.4 MB/s eta 0:00:03
   ---- ----------------------------------- 1.4/13.1 MB 6.0 MB/s eta 0:00:02
   ----- ---------------------------------- 1.8/13.1 MB 6.5 MB/s eta 0:00:02
   ------ --------------------------------- 2.2/13.1 MB 6.7 MB/s eta 0:00:02
   ------- -------------------------------- 2.5/13.1 MB 7.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.9/13.1 MB 6.8 MB/s eta 0:00:02
   --------- ------------------------------ 3.1/13.1 MB 6.7 MB/s eta 0:00:02
   ---------- ----------------------------- 3.3/13.1 MB 6.5 MB/s eta 0:00:02
   ---

In [3]:
import duckdb

# 1. Conteggio istantaneo: quanti viaggi ci sono?
query_count = "SELECT COUNT(*) AS totale_viaggi FROM 'yellow_tripdata_2025-01.parquet'"
print("--- Totale record nel Data Lake ---")
display(duckdb.query(query_count).df())

# 2. La JOIN: uniamo i "Fatti" (viaggi) con le "Dimensioni" (nomi dei quartieri)
query_join = """
    SELECT 
        zone.Borough AS Distretto,
        zone.Zone AS Quartiere,
        COUNT(*) AS Numero_Viaggi,
        ROUND(AVG(fare_amount), 2) AS Corsa_Media_Dollari
    FROM 'yellow_tripdata_2025-01.parquet' AS viaggi
    JOIN 'taxi_zone_lookup.csv' AS zone
      ON viaggi.PULocationID = zone.LocationID
    WHERE fare_amount > 0 
    GROUP BY ALL
    ORDER BY Numero_Viaggi DESC
    LIMIT 10;
"""
print("\n--- Top 10 Quartieri per partenze ---")
display(duckdb.query(query_join).df())

--- Totale record nel Data Lake ---


,totale_viaggi
0,3475226



--- Top 10 Quartieri per partenze ---


,Distretto,Quartiere,Numero_Viaggi,Corsa_Media_Dollari
0,Manhattan,Midtown Center,164142,15.15
1,Manhattan,Upper East Side South,159897,12.23
2,Manhattan,Upper East Side North,151748,12.86
3,Queens,JFK Airport,137448,62.29
4,Manhattan,Times Sq/Theatre District,120061,17.61
5,Manhattan,Penn Station/Madison Sq West,115435,15.50
6,Manhattan,Midtown East,114449,14.79
7,Manhattan,Lincoln Square East,107316,13.78
8,Manhattan,Upper West Side South,93330,13.60
9,Manhattan,Midtown North,92849,15.45


In [ ]:
import pandas as pd
import time

print("--- INIZIO BENCHMARK PANDAS ---")
start_time = time.time()

# 1. Lettura dei file completi in memoria RAM
df_viaggi = pd.read_parquet('yellow_tripdata_2025-01.parquet')
df_zone = pd.read_csv('taxi_zone_lookup.csv')

# 2. Filtraggio: solo corse con costo maggiore di 0
df_viaggi_filtrati = df_viaggi[df_viaggi['fare_amount'] > 0]

# 3. La JOIN (Merge)
df_merged = df_viaggi_filtrati.merge(
    df_zone, 
    left_on='PULocationID', 
    right_on='LocationID'
)

# 4. Aggregazione (Group By)
risultato_pandas = df_merged.groupby(['Borough', 'Zone']).agg(
    Numero_Viaggi=('LocationID', 'count'),
    Corsa_Media_Dollari=('fare_amount', 'mean')
).reset_index()

# 5. Arrotondamento, Ordinamento e Limite
risultato_pandas['Corsa_Media_Dollari'] = risultato_pandas['Corsa_Media_Dollari'].round(2)
risultato_pandas = risultato_pandas.sort_values(by='Numero_Viaggi', ascending=False).head(10)

end_time = time.time()

# Mostriamo il risultato e i tempi
display(risultato_pandas)
tempo_pandas = end_time - start_time
print(f"Tempo di esecuzione Pandas: {tempo_pandas:.2f} secondi")

# Esportiamo il CSV per metterlo su GitHub
risultato_pandas.to_csv("benchmark_risultato_pandas.csv", index=False)

--- INIZIO BENCHMARK PANDAS ---


,Borough,Zone,Numero_Viaggi,Corsa_Media_Dollari
142,Manhattan,Midtown Center,164142,15.15
161,Manhattan,Upper East Side South,159897,12.23
160,Manhattan,Upper East Side North,151748,12.86
201,Queens,JFK Airport,137448,62.29
155,Manhattan,Times Sq/Theatre District,120061,17.61
148,Manhattan,Penn Station/Madison Sq West,115435,15.50
143,Manhattan,Midtown East,114449,14.79
134,Manhattan,Lincoln Square East,107316,13.78
163,Manhattan,Upper West Side South,93330,13.60
144,Manhattan,Midtown North,92849,15.45


Tempo di esecuzione Pandas: 7.84 secondi


In [3]:
import duckdb
import time

print("--- INIZIO BENCHMARK DUCKDB ---")
start_time = time.time()

query_join = """
    SELECT 
        zone.Borough AS Distretto,
        zone.Zone AS Quartiere,
        COUNT(*) AS Numero_Viaggi,
        ROUND(AVG(fare_amount), 2) AS Corsa_Media_Dollari
    FROM 'yellow_tripdata_2025-01.parquet' AS viaggi
    JOIN 'taxi_zone_lookup.csv' AS zone
      ON viaggi.PULocationID = zone.LocationID
    WHERE fare_amount > 0 
    GROUP BY ALL
    ORDER BY Numero_Viaggi DESC
    LIMIT 10;
"""
risultato_duckdb = duckdb.query(query_join).df()

end_time = time.time()

display(risultato_duckdb)
tempo_duckdb = end_time - start_time
print(f"Tempo di esecuzione DuckDB: {tempo_duckdb:.2f} secondi")

# Calcolo del miglioramento percentuale
print(f"\nDuckDB è stato {7.84 / tempo_duckdb:.1f} volte più veloce di Pandas!")

--- INIZIO BENCHMARK DUCKDB ---


,Distretto,Quartiere,Numero_Viaggi,Corsa_Media_Dollari
0,Manhattan,Midtown Center,164142,15.15
1,Manhattan,Upper East Side South,159897,12.23
2,Manhattan,Upper East Side North,151748,12.86
3,Queens,JFK Airport,137448,62.29
4,Manhattan,Times Sq/Theatre District,120061,17.61
5,Manhattan,Penn Station/Madison Sq West,115435,15.50
6,Manhattan,Midtown East,114449,14.79
7,Manhattan,Lincoln Square East,107316,13.78
8,Manhattan,Upper West Side South,93330,13.60
9,Manhattan,Midtown North,92849,15.45


Tempo di esecuzione DuckDB: 0.40 secondi

DuckDB è stato 19.4 volte più veloce di Pandas!


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("--- INIZIO MODELLAZIONE AI (Scikit-Learn) ---")

# 1. Prendiamo il DataFrame pulito generato da DuckDB
df_ml = risultato_duckdb.copy()

# 2. Prepariamo le feature (Normalizzazione con Z-score)
# Selezioniamo le colonne numeriche su cui far imparare l'algoritmo
features = ['Numero_Viaggi', 'Corsa_Media_Dollari']
scaler = StandardScaler()
df_ml_scaled = scaler.fit_transform(df_ml[features])

# 3. Addestriamo il modello K-Means (creiamo 3 cluster/livelli)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ml['Cluster_Livello'] = kmeans.fit_predict(df_ml_scaled)

# Mappiamo i cluster con dei nomi comprensibili (es. Basso, Medio, Alto)
# (Nota: l'assegnazione esatta dipende dai centroidi, questa è una semplificazione visiva)
mappatura_cluster = {0: 'Medio', 1: 'Alto', 2: 'Basso'}
df_ml['Descrizione_Cluster'] = df_ml['Cluster_Livello'].map(mappatura_cluster)

print("\nRisultato del Clustering sui Quartieri:")
display(df_ml[['Distretto', 'Quartiere', 'Numero_Viaggi', 'Corsa_Media_Dollari', 'Descrizione_Cluster']])

# Esportazione dataset
df_ml.to_csv("taxi_zones_clustered.csv", index=False)

--- INIZIO MODELLAZIONE AI (Scikit-Learn) ---


c:\Users\gianl\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1436: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Risultato del Clustering sui Quartieri:


,Distretto,Quartiere,Numero_Viaggi,Corsa_Media_Dollari,Descrizione_Cluster
0,Manhattan,Midtown Center,164142,15.15,Basso
1,Manhattan,Upper East Side South,159897,12.23,Basso
2,Manhattan,Upper East Side North,151748,12.86,Basso
3,Queens,JFK Airport,137448,62.29,Medio
4,Manhattan,Times Sq/Theatre District,120061,17.61,Alto
5,Manhattan,Penn Station/Madison Sq West,115435,15.50,Alto
6,Manhattan,Midtown East,114449,14.79,Alto
7,Manhattan,Lincoln Square East,107316,13.78,Alto
8,Manhattan,Upper West Side South,93330,13.60,Alto
9,Manhattan,Midtown North,92849,15.45,Alto
